In [ ]:
# !pip  install -qU cohere

In [ ]:
import oci
from LoadProperties import LoadProperties
properties=LoadProperties()

# use this for direct cohere model access using cohere-api-key


# # Initialize the Cohere LLM
# YOUR_COHERE_API_KEY = ""

# from langchain.llms import Cohere
# llm = Cohere(cohere_api_key=YOUR_COHERE_API_KEY, temperature=0.7)

# OCI Generative AI service LLM Access

# Import LangChain components
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI

# Initialize the Cohere language model
llm = ChatOCIGenAI(
      model_id='meta.llama-3.3-70b-instruct',
      service_endpoint=properties.getEndpoint(),
      compartment_id=properties.getCompartment(),auth_type='INSTANCE_PRINCIPAL',
      model_kwargs={ "max_tokens": 600},)

### LangChain Expression Language (LCEL) format using the | (pipe) syntax


### Basic Chain  [Basic: One prompt + one LLM]
---
####  Generate a product description for a financial offering

---

**Step 1**: Uses a prompt to extract a detailed product description for a financial offering using the user’s query.

**Step 2**: Converts that detailed info into a single impactful line that can be used in brochures, pitches, or frontline support—demonstrating how AI can abstract complexity into clarity.

This modular approach shows how AI can mimic customer service workflows and enable internal staff or customers to instantly get both depth and clarity— **using chaining.**

---

In [ ]:
from langchain.prompts import PromptTemplate
# from langchain.schema.runnable import RunnableLambda

# Detailed explanation of the financial product
describe_card_prompt = PromptTemplate.from_template(
    "Explain the key features and benefits of the financial product: {card_name}."
)
step1 = {"card_name": lambda x: x["query"]} | describe_card_prompt | llm

# Create a business-friendly one-line summary for executives or customers

summary_tagline_prompt = PromptTemplate.from_template(
    "Summarize this financial product in one compelling business-friendly line:\n\n{text}"
)
step2 = {"text": step1} | summary_tagline_prompt | llm

# Execute the chain

response = step2.invoke({"query": "Premier Gold Credit Card"})

print("Final Output:\n", response.content)


### Sequential Chain   [Multiple inputs/outputs → Multi-step workflows]
---
#### Retail–banking customer interaction
#### CustomerConnect 4-Step Pipeline

- **Intake Chain** captures and condenses the customer’s request.

- **Recommendation Chain** translates that need into 2–3 targeted product suggestions with rationale.

- **Message Chain crafts** a customer-facing pitch, focusing on benefits and clarity.

- **Executive Summary Chain** compiles everything into a one-page overview for senior executives, including suggested next steps and key metrics to track.

In [ ]:
from langchain.llms import Cohere
from langchain.prompts import PromptTemplate


# == Step 1:Capture Customer Need
intake_prompt = PromptTemplate.from_template(
    "You’re a retail-banking assistant. A customer says:\n\n“{customer_query}”\n\nSummarize their primary financial need in one sentence."
)
intake_chain = {"customer_query": lambda x: x["customer_query"]} | intake_prompt | llm

# == Step 2: Recommend Products
recommend_prompt = PromptTemplate.from_template(
    "Based on the need “{need_summary}”, recommend up to three retail-banking products (e.g., savings account, personal loan, credit card). For each, give a one-line rationale."
)
recommend_chain = {"need_summary": intake_chain} | recommend_prompt | llm

# == Step 3: Generate Customer-Facing Message
message_prompt = PromptTemplate.from_template(
    "Craft a concise, benefit-focused message to the customer, weaving in the recommended products:\n\nRecommendations:\n{product_recs}"
)
message_chain = {"product_recs": recommend_chain} | message_prompt | llm

# == Step 4: Draft Executive Summary
exec_prompt = PromptTemplate.from_template(
    """
Prepare a brief executive summary for senior leadership:
• Customer Need: {need_summary}
• Products Recommended: {product_recs}
• Customer Message Preview: {customer_message}
• Next Steps and KPIs to monitor
"""
)
full_chain = {
    "customer_query": lambda x: x["customer_query"],
    "need_summary": intake_chain,
    "product_recs": recommend_chain,
    "customer_message": message_chain
} | exec_prompt | llm

# == Execute the 4-step chain

output = full_chain.invoke({
    "customer_query": "I want to consolidate my credit-card debt and start earning travel rewards."
})

# == Display the generated output

# print("Final Output:\n", output.content)

from IPython.display import Markdown, display
display(Markdown( "Final Output:\n " + output.content))
